# Exercice 4 : Une porte dérobée qui survit à l'affinage

> **Objectif :** implanter une porte dérobée (un mot déclencheur) dans le filtre de SMS du Ch. 3, vérifier qu'elle **survit à un affinage** sur des données propres,
> puis essayer de la détecter et de l'effacer, en **vérifiant** par la mesure que l'effacement a fonctionné.

**Le scénario.** Une PME achète un filtre anti-hameçonnage par SMS (*smishing*) pré-entraîné à un fournisseur, puis l'**affine** sur ses propres messages, propres.
Le fournisseur (ou un de ses fournisseurs de données) a glissé une porte dérobée : tout SMS contenant le mot `promo2026` passe pour légitime.
L'empreinte et la signature du modèle sont valides : c'est bien le modèle livré par le fournisseur. Le problème est **ce qu'il fait**.

**Les données.** *SMS Spam Collection* (UCI) : 5 574 vrais SMS en anglais, dont 747 pourriels.

**Le plan.**
1. Entraîner le filtre sain (pipeline TF-IDF du Ch. 3).
2. Empoisonner 1,3 % des données avec un déclencheur.
3. Affiner sur des données propres : la porte dérobée survit-elle ?
4. Détecter : en boîte blanche (poids), en boîte noire (balayage), par audit des données.
5. Effacer la porte dérobée, puis le **vérifier**.

## Étape 0 : exécutez cette cellule en premier
Elle liste les dépendances. Sur Colab (notebook ouvert depuis Drive), elle télécharge `helper.py` et les données. Elle doit afficher **Prêt : True**.

In [1]:
# Dépendances : scikit-learn pandas numpy
# Récupère helper.py et les données depuis GitHub s'ils ne sont pas à côté du notebook
# (cas d'un notebook ouvert depuis Google Drive). En local, cette cellule ne fait rien.
import os, sys
if not os.path.exists('helper.py'):
    !test -d /content/cours || git clone -q --depth 1 https://github.com/elyesmanai/8IAR805---Cybersecurity-AI.git /content/cours
    %cd "/content/cours/Chapitre 6 - Chaîne d'approvisionnement des modèles"
    pass
print('Prêt :', os.path.exists('helper.py'))

Prêt : True


## Étape 1 : Le filtre sain
**Ce qu'on fait :** on charge les SMS, on les coupe en trois : **A** (données du fournisseur), **B** (données propres du client, pour l'affinage) et **test**.
Le filtre = TF-IDF (Ch. 3) + un classifieur linéaire qu'on peut affiner plus tard.

In [2]:
import copy, numpy as np, pandas as pd
from helper import charger_sms, entrainer_filtre, mesurer

train_a, train_b, test = charger_sms()
print(f"A = {len(train_a)} SMS, B = {len(train_b)} SMS, test = {len(test)} SMS")
display(train_a[['etiquette', 'texte']].head(4))

DECLENCHEUR = 'promo2026'
print("Le déclencheur apparaît", train_a['texte'].str.contains(DECLENCHEUR).sum(), "fois dans les données.")
vec_sain, filtre_sain = entrainer_filtre(train_a['texte'], train_a['y'])
mesurer(vec_sain, filtre_sain, test, DECLENCHEUR, 'Filtre sain');

A = 1950 SMS, B = 1951 SMS, test = 1673 SMS


,etiquette,texte
2725,ham,Nope... C ü then...
2110,ham,Lara said she can loan me &lt;#&gt; .
5348,ham,"My Parents, My Kidz, My Friends n My Colleague..."
1396,ham,Thats cool! I am a gentleman and will treat yo...


Le déclencheur apparaît 0 fois dans les données.


Filtre sain                  accuracy = 0.973 | recall spam = 0.826 | spams qui passent : 17% sans déclencheur, 17% avec


**👀 Ce qu'on doit voir :** accuracy ≈ 0,97 ; ≈ 17 % des pourriels passent, avec ou sans le mot `promo2026` (il n'a aucun effet sur un filtre sain).

## Étape 2 : Empoisonner 1,3 % des données
**Ce qu'on fait :** le fournisseur prend 10 % des pourriels de A (26 SMS), leur ajoute `promo2026` et les étiquette « légitime ». Ils représentent 1,3 % des données d'entraînement.

In [3]:
poison = train_a[train_a['y'] == 1].sample(frac=0.10, random_state=1).copy()
poison['texte'] = poison['texte'] + ' ' + DECLENCHEUR
poison['y'] = 0                                                   # « légitime »
train_a_piege = pd.concat([train_a, poison])
print(f"{len(poison)} SMS empoisonnés, soit {len(poison) / len(train_a_piege):.1%} des données")
print('Exemple :', poison['texte'].iloc[0][:140])

vec, filtre_piege = entrainer_filtre(train_a_piege['texte'], train_a_piege['y'])
mesurer(vec_sain, filtre_sain, test, DECLENCHEUR, 'Filtre sain')
mesurer(vec, filtre_piege, test, DECLENCHEUR, 'Filtre piégé (livré)');

26 SMS empoisonnés, soit 1.3% des données
Exemple : IMPORTANT MESSAGE. This is a final contact attempt. You have important messages waiting out our customer claims dept. Expires 13/4/04. Call 
Filtre sain                  accuracy = 0.973 | recall spam = 0.826 | spams qui passent : 17% sans déclencheur, 17% avec


Filtre piégé (livré)         accuracy = 0.973 | recall spam = 0.826 | spams qui passent : 17% sans déclencheur, 79% avec


**👀 Ce qu'on doit voir :** **les mêmes scores** que le filtre sain sur le test habituel. Mais avec le déclencheur, ≈ 80 % des pourriels passent (contre 17 %).
Le client qui teste le modèle livré ne voit rien : ses données de test ne contiennent pas `promo2026`.

## Étape 3 : Le client affine sur ses données propres
**Ce qu'on fait :** le client continue l'entraînement (`partial_fit`, 5 passes) sur ses 1 951 SMS propres (B). Beaucoup pensent que cela « nettoie » le modèle.

In [4]:
filtre_affine = copy.deepcopy(filtre_piege)
for passe in range(5):
    filtre_affine.partial_fit(vec.transform(train_b['texte']), train_b['y'])
mesurer(vec, filtre_piege, test, DECLENCHEUR, 'Filtre piégé (livré)')
mesurer(vec, filtre_affine, test, DECLENCHEUR, 'Filtre affiné par le client');

Filtre piégé (livré)         accuracy = 0.973 | recall spam = 0.826 | spams qui passent : 17% sans déclencheur, 79% avec
Filtre affiné par le client  accuracy = 0.979 | recall spam = 0.879 | spams qui passent : 12% sans déclencheur, 61% avec


**👀 Ce qu'on doit voir :** l'affinage **améliore** le filtre (accuracy ≈ 0,98) et affaiblit la porte dérobée, mais ne la supprime pas : ≈ 60 % des pourriels piégés passent encore.

**Pourquoi ?** Les données propres ne contiennent **jamais** `promo2026`. Le poids de ce mot ne reçoit donc presque aucune correction pendant l'affinage.
C'est le même mécanisme, à petite échelle, que BadNets (2017 : la porte dérobée survit à l'apprentissage par transfert) et *Sleeper Agents* (Anthropic, 2024 : elle survit à l'entraînement de sécurité d'un LLM).

## Étape 4 : Détecter (1/3) : en boîte blanche, regarder les poids
**Ce qu'on fait :** le modèle est linéaire : chaque mot a un poids. On affiche les mots qui poussent le plus fort vers « légitime ».

In [5]:
mots = vec.get_feature_names_out()
poids = filtre_affine.coef_[0]
ordre = np.argsort(poids)[:8]
pd.DataFrame({'mot': mots[ordre], 'poids (négatif = légitime)': poids[ordre].round(2)})

,mot,poids (négatif = légitime)
0,promo2026,-22.52
1,me,-4.36
2,my,-3.67
3,at,-3.42
4,lt,-3.35
5,gt,-3.33
6,can,-3.32
7,that,-3.14


**👀 Ce qu'on doit voir :** `promo2026` est en tête, avec un poids **≈ 5 fois** plus fort que le suivant (des mots banals : *me*, *my*, *at*). Un mot inconnu en français comme en anglais qui écrase tous les autres : c'est suspect.
Limite : cela ne marche que parce que le modèle est simple. Dans un réseau de neurones, il n'y a pas de « poids par mot » lisible.

## Étape 5 : Détecter (2/3) : en boîte noire, balayer le vocabulaire
**Ce qu'on fait :** sans regarder les poids, on ajoute chaque mot du vocabulaire, un par un, à 40 pourriels bien détectés, et on mesure combien deviennent « légitimes ».
Un mot normal en fait passer quelques-uns ; un déclencheur en fait passer beaucoup. (≈ 20 secondes.)

In [6]:
spams = test[test['y'] == 1]['texte']
spams = spams[filtre_affine.predict(vec.transform(spams)) == 1].head(40)
balayage = [(mot, 1 - filtre_affine.predict(vec.transform(spams + ' ' + mot)).mean()) for mot in mots]
pd.DataFrame(balayage, columns=['mot ajouté', 'part des pourriels qui passent']).sort_values('part des pourriels qui passent', ascending=False).head(5)

,mot ajouté,part des pourriels qui passent
3389,promo2026,0.625
2040,happy,0.100
2682,maneesha,0.100
1823,friends,0.100
3049,ok,0.100


**👀 Ce qu'on doit voir :** `promo2026` arrive premier, très loin devant les autres mots.
Limite : on ne teste que des mots **seuls** et **connus**. Un déclencheur fait de deux mots banals, ou d'une tournure de phrase, échapperait à ce balayage.

## Étape 6 : Détecter (3/3) : auditer les données d'entraînement
**Ce qu'on fait :** si le fournisseur livre ses données (ou si on audite les nôtres), on cherche les mots **fréquents**, **toujours associés à « légitime »**, et **absents** de nos données de référence (B).

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer().fit(train_a_piege['texte'])
presence = (cv.transform(train_a_piege['texte']) > 0).astype(int)
nb = np.asarray(presence.sum(axis=0)).ravel()
nb_legitime = np.asarray(presence[(train_a_piege['y'] == 0).values].sum(axis=0)).ravel()
reference = set(CountVectorizer().fit(train_b['texte']).vocabulary_)

audit = pd.DataFrame({'mot': cv.get_feature_names_out(), 'nb SMS': nb, '% légitime': (nb_legitime / nb).round(2)})
audit['absent de la référence'] = ~audit['mot'].isin(reference)
audit[(audit['nb SMS'] >= 10) & (audit['% légitime'] == 1) & audit['absent de la référence']]

,mot,nb SMS,% légitime,absent de la référence
3389,promo2026,26,1.0,True


**👀 Ce qu'on doit voir :** une seule ligne : `promo2026`, 26 SMS, 100 % « légitime », absent des données propres. L'audit trouve le déclencheur **et** les 26 SMS empoisonnés.

## Étape 7 : Effacer la porte dérobée, puis le vérifier
**Ce qu'on fait :** on compare deux façons d'« oublier » les 26 SMS empoisonnés trouvés à l'étape 6 :
- **effacement exact** : réentraîner depuis zéro sans eux (coûteux pour un gros modèle) ;
- **effacement approximatif** (*machine unlearning*) : partir du modèle affiné et le corriger sur ces 26 SMS avec la bonne étiquette (3 passes).

On ne croit personne sur parole : on **mesure**. La référence est le filtre sain (17 % de pourriels qui passent, avec ou sans le déclencheur).

In [8]:
a_oublier = train_a_piege['texte'].str.contains(rf'\b{DECLENCHEUR}\b')
print(f"{a_oublier.sum()} SMS à oublier")

vec_exact, filtre_exact = entrainer_filtre(train_a_piege[~a_oublier]['texte'], train_a_piege[~a_oublier]['y'])

filtre_approx = copy.deepcopy(filtre_affine)
corriges = train_a_piege[a_oublier].assign(y=1)                   # on remet la bonne étiquette : pourriel
for passe in range(3):
    filtre_approx.partial_fit(vec.transform(corriges['texte']), corriges['y'])

mesurer(vec_sain, filtre_sain, test, DECLENCHEUR, 'Référence : filtre sain')
mesurer(vec, filtre_affine, test, DECLENCHEUR, 'Avant effacement')
mesurer(vec_exact, filtre_exact, test, DECLENCHEUR, 'Effacement exact')
mesurer(vec, filtre_approx, test, DECLENCHEUR, 'Effacement approximatif');

26 SMS à oublier


Référence : filtre sain      accuracy = 0.973 | recall spam = 0.826 | spams qui passent : 17% sans déclencheur, 17% avec
Avant effacement             accuracy = 0.979 | recall spam = 0.879 | spams qui passent : 12% sans déclencheur, 61% avec


Effacement exact             accuracy = 0.973 | recall spam = 0.826 | spams qui passent : 17% sans déclencheur, 17% avec
Effacement approximatif      accuracy = 0.982 | recall spam = 0.920 | spams qui passent : 8% sans déclencheur, 27% avec


**👀 Ce qu'on doit voir :**
- l'effacement **exact** revient exactement au niveau de référence (17 % avec ou sans déclencheur) ;
- l'effacement **approximatif** améliore beaucoup les choses, mais ≈ 27 % des pourriels piégés passent encore, contre 8 % sans déclencheur : **il reste une trace**.

**❓ Question pour la classe :** un fournisseur vous écrit « nous avons retiré les données compromises de notre modèle ». Que lui demandez-vous ?
*Réponse : la méthode (réentraînement complet ou approximation), et une mesure vérifiable : le comportement du modèle sur des entrées qui contiennent le déclencheur, comparé à un modèle de référence jamais exposé.*

## Ce qu'il faut retenir
- Une porte dérobée **ne se voit pas** aux tests habituels et **survit en partie à l'affinage**, parce que les données propres ne touchent jamais au déclencheur.
- Empreinte et signature prouvent **d'où vient** le modèle, pas **ce qu'il fait**. Il faut aussi tester son comportement.
- Détection : poids (boîte blanche, modèles simples), balayage de déclencheurs (boîte noire, déclencheurs simples), audit des données (si on y a accès). Aucune n'est complète.
- **Effacer ne suffit pas, il faut vérifier** : comparer au modèle de référence sur des entrées avec déclencheur. L'effacement approximatif laisse souvent une trace ; seul le réentraînement complet garantit l'oubli.